# Experiment: Inspect MultiLepPAT GEN-RECO Matching

This notebook inspects a `MultiLepPAT` ntuple with emphasis on five questions:

1. Is the GEN-level event content kept even when reco selection is sparse?
2. How often do muons use PAT GEN refs versus the chi2 fallback?
3. How are phi-daughter kaons associated to PVs and matched back into `MC_GenPart_*`?
4. What decay signatures and mother chains are visible from the stored flat GEN collection?
5. What changes when `RequireAcceptedCandidatesForMonteCarloTree` is enabled on the DPS validation sample?

The notebook is written to prefer the recent DPS validation ntuples, and falls back to the documented validation ntuple if needed.


## Notebook Outline

- Configure the ntuple path and open `mkcands/X_data`.
- Inspect branch inventory for GEN, muon matching, and phi-kaon matching content.
- Summarize event-level GEN retention, muon match provenance, and phi-kaon match coverage.
- Inspect one event in detail, joining reco muons and fitted phi-daughter kaons to matched GEN particles.
- Rebuild decay chains from `MC_GenPart_motherGenIdx`, including the matched GEN phi for kaons.
- Compare the paired DPS validation ntuples with `RequireAcceptedCandidatesForMonteCarloTree=False/True`.


In [1]:
from pathlib import Path
from collections import Counter
from math import hypot, isfinite

import ROOT
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT.gROOT.SetBatch(True)

CANDIDATE_NTUPLES = [
    Path('/tmp/conf_dps120_switch_false_numEvent10000.root'),
    Path('/tmp/conf_dps120_switch_true_numEvent10000.root'),
    Path('/tmp/dps_phi_vertexdiag_numEvent1000.root'),
    Path('/tmp/dps_phi_kaonmatch_1000_numEvent1000.root'),
    Path('/tmp/dps_phi_kaonmatch_numEvent100.root'),
    Path('/eos/user/c/chiw/JpsiJpsiPhi/CMSSW_15_0_15_JpsiJpsiPhi_refactor/src/HeavyFlavorAnalysis/TPS-Onia2MuMu/dps_phi_kaonmatch_1000_numEvent1000.root'),
]
RETENTION_VALIDATION_NTUPLES = {
    'keep_all_mc_events': Path('/tmp/conf_dps120_switch_false_numEvent10000.root'),
    'require_candidates': Path('/tmp/conf_dps120_switch_true_numEvent10000.root'),
}
VALIDATION_NTUPLE = Path('/tmp/maxevents_fix_check_numEvent5.root')
TREE_PATH = 'mkcands/X_data'

NTUPLE_PATH = next((path for path in CANDIDATE_NTUPLES if path.exists()), VALIDATION_NTUPLE)
if not NTUPLE_PATH.exists():
    raise FileNotFoundError('Set NTUPLE_PATH to a valid MultiLepPAT ntuple before running the notebook.')

root_file = ROOT.TFile.Open(str(NTUPLE_PATH), 'READ')
tree = root_file.Get(TREE_PATH)
if not tree:
    raise RuntimeError(f'Could not find {TREE_PATH} in {NTUPLE_PATH}')

print(f'Using ntuple: {NTUPLE_PATH}')
print(f'Entries in {TREE_PATH}: {tree.GetEntries()}')


Using ntuple: /eos/user/c/chiw/JpsiJpsiPhi/CMSSW_15_0_15_JpsiJpsiPhi_refactor/src/HeavyFlavorAnalysis/TPS-Onia2MuMu/dps_phi_kaonmatch_1000_numEvent1000.root
Entries in mkcands/X_data: 945


In [2]:
for branch in tree.GetListOfBranches():
    print(branch.GetName())

TrigRes
TrigNames
MatchJpsiTriggerNames
MatchUpsTriggerNames
L1TrigRes
evtNum
runNum
lumiNum
nGoodPrimVtx
priVtxX
priVtxY
priVtxZ
priVtxXE
priVtxYE
priVtxZE
priVtxChiNorm
priVtxChi
priVtxCL
PriVtxXCorrX
PriVtxXCorrY
PriVtxXCorrZ
PriVtxXCorrEX
PriVtxXCorrEY
PriVtxXCorrEZ
PriVtxXCorrC2
PriVtxXCorrCL
nRecVtx
RecVtx_x
RecVtx_y
RecVtx_z
RecVtx_xErr
RecVtx_yErr
RecVtx_zErr
RecVtx_chi2
RecVtx_ndof
RecVtx_vtxProb
RecVtx_nTracks
nMu
muPx
muPy
muPz
muD0
muD0E
muDz
muChi2
muGlChi2
mufHits
muFirstBarrel
muFirstEndCap
muDzVtx
muDxyVtx
muNDF
muGlNDF
muPhits
muShits
muGlMuHits
muType
muQual
muTrack
muCharge
muIsoratio
munMatchedSeg
muIsGoodSoftMuonNewIlseMod
muIsGoodSoftMuonNewIlse
muIsGoodLooseMuonNew
muIsGoodLooseMuon
muIsGoodTightMuon
muIsGlobalMuon
muIsPatLooseMuon
muIsPatTightMuon
muIsPatSoftMuon
muIsPatMediumMuon
muFromPV
muPVAssocQuality
muPxErr
muPyErr
muPzErr
muPtErr
muVertexId
muDzAssocPV
muDxyAssocPV
muFromPVAssocPV
muPdgId
muPackedMatchIdx
muPackedMatchMethod
muPackedMatchVectorRelP
muPac

In [3]:
branches = [branch.GetName() for branch in tree.GetListOfBranches()]
focus_branches = [
    name for name in branches
    if name.startswith('MC_GenPart')
    or name in {
        'Pri_fitValid',
        'Pri_fitPass',
        'Pri_assocPVPass',
        'Pri_assocPVIdx',
        'Pri_trackPVPass',
        'Pri_passAny',
        'Pri_maxAbsDzPV',
        'Pri_maxAbsDxyPV',
        'Phi_fitPass',
        'Phi_commonAssocPVPass',
        'Phi_commonAssocPVIdx',
        'Phi_trackPVPass',
        'Phi_vertexCriteriaPass',
        'Phi_maxAbsDzPV',
        'Phi_maxAbsDxyPV',
    }
    or name.startswith('muGenMatch')
    or name.startswith('Phi_K_1_genMatch')
    or name.startswith('Phi_K_2_genMatch')
    or name in {
        'muVertexId',
        'Phi_K_1_vertexId', 'Phi_K_2_vertexId',
        'Phi_K_1_hasAssocPV', 'Phi_K_2_hasAssocPV',
        'Phi_K_1_passDzPV', 'Phi_K_2_passDzPV',
        'Phi_K_1_passDxyPV', 'Phi_K_2_passDxyPV',
        'Phi_K_1_passTrackPV', 'Phi_K_2_passTrackPV',
        'Phi_K_1_dzPV', 'Phi_K_1_dxyPV', 'Phi_K_1_dzAssocPV', 'Phi_K_1_dxyAssocPV',
        'Phi_K_2_dzPV', 'Phi_K_2_dxyPV', 'Phi_K_2_dzAssocPV', 'Phi_K_2_dxyAssocPV',
        'Phi_K_1_Idx', 'Phi_K_2_Idx',
        'Jpsi_1_mu_1_Idx', 'Jpsi_1_mu_2_Idx',
        'Jpsi_2_mu_1_Idx', 'Jpsi_2_mu_2_Idx',
        'Ups_mu_1_Idx', 'Ups_mu_2_Idx',
    }
]

for name in focus_branches:
    print(name)


muVertexId
muGenMatchIdx
muGenMatchSource
muGenMatchChi2
Jpsi_1_mu_1_Idx
Jpsi_1_mu_2_Idx
Jpsi_2_mu_1_Idx
Jpsi_2_mu_2_Idx
Phi_K_1_Idx
Phi_K_2_Idx
Pri_fitValid
Pri_fitPass
Pri_assocPVPass
Pri_assocPVIdx
Pri_trackPVPass
Pri_passAny
Pri_maxAbsDzPV
Pri_maxAbsDxyPV
Phi_K_1_vertexId
Phi_K_1_dzPV
Phi_K_1_dxyPV
Phi_K_1_dzAssocPV
Phi_K_1_dxyAssocPV
Phi_K_1_genMatchIdx
Phi_K_1_genMatchSource
Phi_K_1_genMatchChi2
Phi_K_2_vertexId
Phi_K_2_dzPV
Phi_K_2_dxyPV
Phi_K_2_dzAssocPV
Phi_K_2_dxyAssocPV
Phi_K_2_genMatchIdx
Phi_K_2_genMatchSource
Phi_K_2_genMatchChi2
Ups_mu_1_Idx
Ups_mu_2_Idx
MC_GenPart_pdgId
MC_GenPart_status
MC_GenPart_motherPdgId
MC_GenPart_motherGenIdx
MC_GenPart_handleIndex
MC_GenPart_px
MC_GenPart_py
MC_GenPart_pz
MC_GenPart_mass
MC_GenPart_pt
MC_GenPart_eta
MC_GenPart_phi


In [6]:
PDG_LABELS = {
    443: 'J/psi',
    -443: 'J/psi',
    553: 'Upsilon',
    -553: 'Upsilon',
    333: 'phi',
    -333: 'phi',
    13: 'mu-',
    -13: 'mu+',
    321: 'K+',
    -321: 'K-',
}

MU_MATCH_SOURCE_LABELS = {0: 'unmatched', 1: 'patRef', 2: 'chi2Fallback'}
KAON_MATCH_SOURCE_LABELS = {0: 'unmatched', 1: 'phiMotherChi2', 2: 'chi2Fallback'}

def pdg_label(pdg_id):
    return PDG_LABELS.get(int(pdg_id), str(int(pdg_id)))

def as_list(vec):
    return list(vec) if vec is not None else []

def maybe_frame(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows

def walk_mother_chain(gen_rows, start_idx, max_depth=8):
    chain = []
    idx = int(start_idx)
    seen = set()
    while 0 <= idx < len(gen_rows) and idx not in seen and len(chain) < max_depth:
        seen.add(idx)
        row = gen_rows[idx]
        chain.append(f"{idx}:{row['particle']}")
        idx = int(row['mother_idx'])
    return ' <- '.join(chain)

def find_ancestor_idx(gen_rows, start_idx, target_abs_pdg_id):
    idx = int(start_idx)
    seen = set()
    while 0 <= idx < len(gen_rows) and idx not in seen:
        seen.add(idx)
        row = gen_rows[idx]
        if abs(int(row['pdg_id'])) == target_abs_pdg_id:
            return idx
        idx = int(row['mother_idx'])
    return -1

def first_entry_with_phi(tree):
    for entry in range(tree.GetEntries()):
        tree.GetEntry(entry)
        if hasattr(tree, 'Phi_K_1_Idx') and tree.Phi_K_1_Idx.size() > 0:
            return entry
    return 0

def event_snapshot(tree, entry):
    tree.GetEntry(entry)

    gen_rows = []
    for idx, pdg_id in enumerate(as_list(tree.MC_GenPart_pdgId)):
        gen_rows.append({
            'gen_idx': idx,
            'handle_index': int(tree.MC_GenPart_handleIndex[idx]) if hasattr(tree, 'MC_GenPart_handleIndex') else -1,
            'pdg_id': int(pdg_id),
            'particle': pdg_label(pdg_id),
            'status': int(tree.MC_GenPart_status[idx]),
            'mother_idx': int(tree.MC_GenPart_motherGenIdx[idx]),
            'mother_pdg_id': int(tree.MC_GenPart_motherPdgId[idx]),
            'mother': pdg_label(tree.MC_GenPart_motherPdgId[idx]),
            'pt': float(tree.MC_GenPart_pt[idx]),
            'eta': float(tree.MC_GenPart_eta[idx]),
            'phi': float(tree.MC_GenPart_phi[idx]),
            'mass': float(tree.MC_GenPart_mass[idx]),
        })

    mu_rows = []
    mu_px = as_list(tree.muPx)
    mu_py = as_list(tree.muPy)
    mu_pz = as_list(tree.muPz)
    mu_charge = as_list(tree.muCharge)
    mu_gen_idx = as_list(tree.muGenMatchIdx) if hasattr(tree, 'muGenMatchIdx') else [-1] * len(mu_px)
    mu_gen_source = as_list(tree.muGenMatchSource) if hasattr(tree, 'muGenMatchSource') else [0] * len(mu_px)
    mu_gen_chi2 = as_list(tree.muGenMatchChi2) if hasattr(tree, 'muGenMatchChi2') else [-1.0] * len(mu_px)

    for idx, px in enumerate(mu_px):
        match_idx = int(mu_gen_idx[idx]) if idx < len(mu_gen_idx) else -1
        matched_gen = gen_rows[match_idx] if 0 <= match_idx < len(gen_rows) else None
        mu_rows.append({
            'mu_idx': idx,
            'charge': int(mu_charge[idx]),
            'pt': float(hypot(px, mu_py[idx])),
            'eta': float(0.5 * ROOT.TMath.Log((hypot(px, mu_py[idx], mu_pz[idx]) + mu_pz[idx]) / (hypot(px, mu_py[idx], mu_pz[idx]) - mu_pz[idx]))) if (hypot(px, mu_py[idx], mu_pz[idx]) - mu_pz[idx]) != 0 else float('inf') * (1 if mu_pz[idx] >= 0 else -1),
            'phi': float(ROOT.TMath.ATan2(mu_py[idx], px)),
            'vertexId': int(tree.muVertexId[idx]) if hasattr(tree, 'muVertexId') else -1,
            'match_idx': match_idx,
            'match_source': MU_MATCH_SOURCE_LABELS.get(int(mu_gen_source[idx]), int(mu_gen_source[idx])),
            'match_chi2': float(mu_gen_chi2[idx]) if idx < len(mu_gen_chi2) and isfinite(float(mu_gen_chi2[idx])) else None,
            'matched_pdg_id': matched_gen['pdg_id'] if matched_gen else None,
            'matched_particle': matched_gen['particle'] if matched_gen else None,
            'matched_mother': matched_gen['mother'] if matched_gen else None,
            'matched_mother_idx': matched_gen['mother_idx'] if matched_gen else None,
            'matched_gen_pt': matched_gen['pt'] if matched_gen else None,
        })

    phi_rows = []
    phi_specs = [
        ('K1', 'Phi_K_1_Idx', 'Phi_K_1_pt', 'Phi_K_1_eta', 'Phi_K_1_phi', 'Phi_K_1_vertexId', 'Phi_K_1_fromPV', 'Phi_K_1_pvAssocQuality',
         'Phi_K_1_hasAssocPV', 'Phi_K_1_passDzPV', 'Phi_K_1_passDxyPV', 'Phi_K_1_passTrackPV',
         'Phi_K_1_dzPV', 'Phi_K_1_dxyPV', 'Phi_K_1_dzAssocPV', 'Phi_K_1_dxyAssocPV', 'Phi_K_1_genMatchIdx', 'Phi_K_1_genMatchSource', 'Phi_K_1_genMatchChi2'),
        ('K2', 'Phi_K_2_Idx', 'Phi_K_2_pt', 'Phi_K_2_eta', 'Phi_K_2_phi', 'Phi_K_2_vertexId', 'Phi_K_2_fromPV', 'Phi_K_2_pvAssocQuality',
         'Phi_K_2_hasAssocPV', 'Phi_K_2_passDzPV', 'Phi_K_2_passDxyPV', 'Phi_K_2_passTrackPV',
         'Phi_K_2_dzPV', 'Phi_K_2_dxyPV', 'Phi_K_2_dzAssocPV', 'Phi_K_2_dxyAssocPV', 'Phi_K_2_genMatchIdx', 'Phi_K_2_genMatchSource', 'Phi_K_2_genMatchChi2'),
    ]

    phi_fit_pass = as_list(getattr(tree, 'Phi_fitPass', []))
    phi_common_assoc_pass = as_list(getattr(tree, 'Phi_commonAssocPVPass', []))
    phi_common_assoc_idx = as_list(getattr(tree, 'Phi_commonAssocPVIdx', []))
    phi_track_pv_pass = as_list(getattr(tree, 'Phi_trackPVPass', []))
    phi_vertex_pass = as_list(getattr(tree, 'Phi_vertexCriteriaPass', []))
    phi_max_abs_dz = as_list(getattr(tree, 'Phi_maxAbsDzPV', []))
    phi_max_abs_dxy = as_list(getattr(tree, 'Phi_maxAbsDxyPV', []))

    for role, idx_name, pt_name, eta_name, phi_name, vertex_name, from_pv_name, pv_quality_name, has_assoc_name, pass_dz_name, pass_dxy_name, pass_track_name, dzpv_name, dxypv_name, dzassoc_name, dxyassoc_name, gen_idx_name, gen_source_name, gen_chi2_name in phi_specs:
        idx_vec = as_list(getattr(tree, idx_name, []))
        pt_vec = as_list(getattr(tree, pt_name, []))
        eta_vec = as_list(getattr(tree, eta_name, []))
        phi_vec = as_list(getattr(tree, phi_name, []))
        vertex_vec = as_list(getattr(tree, vertex_name, []))
        from_pv_vec = as_list(getattr(tree, from_pv_name, []))
        pv_quality_vec = as_list(getattr(tree, pv_quality_name, []))
        has_assoc_vec = as_list(getattr(tree, has_assoc_name, []))
        pass_dz_vec = as_list(getattr(tree, pass_dz_name, []))
        pass_dxy_vec = as_list(getattr(tree, pass_dxy_name, []))
        pass_track_vec = as_list(getattr(tree, pass_track_name, []))
        dzpv_vec = as_list(getattr(tree, dzpv_name, []))
        dxypv_vec = as_list(getattr(tree, dxypv_name, []))
        dzassoc_vec = as_list(getattr(tree, dzassoc_name, []))
        dxyassoc_vec = as_list(getattr(tree, dxyassoc_name, []))
        gen_idx_vec = as_list(getattr(tree, gen_idx_name, []))
        gen_source_vec = as_list(getattr(tree, gen_source_name, []))
        gen_chi2_vec = as_list(getattr(tree, gen_chi2_name, []))

        for cand_idx, track_idx in enumerate(idx_vec):
            match_idx = int(gen_idx_vec[cand_idx]) if cand_idx < len(gen_idx_vec) else -1
            matched_gen = gen_rows[match_idx] if 0 <= match_idx < len(gen_rows) else None
            matched_phi_idx = find_ancestor_idx(gen_rows, match_idx, 333) if matched_gen else -1
            phi_rows.append({
                'phi_candidate': cand_idx,
                'kaon_role': role,
                'track_idx': int(track_idx),
                'phi_fit_pass': int(phi_fit_pass[cand_idx]) if cand_idx < len(phi_fit_pass) else -1,
                'phi_commonAssocPVPass': int(phi_common_assoc_pass[cand_idx]) if cand_idx < len(phi_common_assoc_pass) else -1,
                'phi_commonAssocPVIdx': int(phi_common_assoc_idx[cand_idx]) if cand_idx < len(phi_common_assoc_idx) else -1,
                'phi_trackPVPass': int(phi_track_pv_pass[cand_idx]) if cand_idx < len(phi_track_pv_pass) else -1,
                'phi_vertexCriteriaPass': int(phi_vertex_pass[cand_idx]) if cand_idx < len(phi_vertex_pass) else -1,
                # 'phi_maxAbsDzPV': float(phi_max_abs_dz[cand_idx]) if cand_idx < len(phi_max_abs_dz) else None,
                # 'phi_maxAbsDxyPV': float(phi_max_abs_dxy[cand_idx]) if cand_idx < len(phi_max_abs_dxy) else None,
                'pt': float(pt_vec[cand_idx]) if cand_idx < len(pt_vec) else None,
                'eta': float(eta_vec[cand_idx]) if cand_idx < len(eta_vec) else None,
                'phi': float(phi_vec[cand_idx]) if cand_idx < len(phi_vec) else None,
                'vertexId': int(vertex_vec[cand_idx]) if cand_idx < len(vertex_vec) else -1,
                'fromPV': int(from_pv_vec[cand_idx]) if cand_idx < len(from_pv_vec) else -1,
                'pvAssocQuality': int(pv_quality_vec[cand_idx]) if cand_idx < len(pv_quality_vec) else -1,
                'hasAssocPV': int(has_assoc_vec[cand_idx]) if cand_idx < len(has_assoc_vec) else -1,
                # 'passDzPV': int(pass_dz_vec[cand_idx]) if cand_idx < len(pass_dz_vec) else -1,
                # 'passDxyPV': int(pass_dxy_vec[cand_idx]) if cand_idx < len(pass_dxy_vec) else -1,
                'passTrackPV': int(pass_track_vec[cand_idx]) if cand_idx < len(pass_track_vec) else -1,
                # 'dzPV': float(dzpv_vec[cand_idx]) if cand_idx < len(dzpv_vec) else None,
                # 'dxyPV': float(dxypv_vec[cand_idx]) if cand_idx < len(dxypv_vec) else None,
                # 'dzAssocPV': float(dzassoc_vec[cand_idx]) if cand_idx < len(dzassoc_vec) else None,
                # 'dxyAssocPV': float(dxyassoc_vec[cand_idx]) if cand_idx < len(dxyassoc_vec) else None,
                'gen_match_idx': match_idx,
                'gen_match_source': KAON_MATCH_SOURCE_LABELS.get(int(gen_source_vec[cand_idx]), int(gen_source_vec[cand_idx])) if cand_idx < len(gen_source_vec) else 'unmatched',
                'gen_match_chi2': float(gen_chi2_vec[cand_idx]) if cand_idx < len(gen_chi2_vec) and isfinite(float(gen_chi2_vec[cand_idx])) else None,
                'matched_particle': matched_gen['particle'] if matched_gen else None,
                'matched_phi_gen_idx': matched_phi_idx,
                'matched_phi_particle': gen_rows[matched_phi_idx]['particle'] if 0 <= matched_phi_idx < len(gen_rows) else None,
                'gen_chain': walk_mother_chain(gen_rows, match_idx) if matched_gen else None,
            })

    return {
        'entry': entry,
        'run': int(tree.runNum),
        'lumi': int(tree.lumiNum),
        'event': int(tree.evtNum),
        'nMu': int(tree.nMu),
        'gen_rows': gen_rows,
        'mu_rows': mu_rows,
        'phi_rows': phi_rows,
    }


In [8]:
summary = {
    'entries': int(tree.GetEntries()),
    'events_gen_nonempty': 0,
    'events_with_muons': 0,
    'events_with_patref_match': 0,
    'events_with_chi2_fallback': 0,
    'events_with_phi_candidates': 0,
    'events_with_matched_phi_kaons': 0,
    'events_with_phi_common_assoc_pv': 0,
    'events_with_phi_track_pv_pass': 0,
    'events_with_phi_vertex_criteria_pass': 0,
    'total_muons': 0,
    'total_matched_muons': 0,
    'total_patref_matches': 0,
    'total_chi2_fallback_matches': 0,
    'total_phi_kaons': 0,
    'total_matched_phi_kaons': 0,
    'total_phi_candidates': 0,
    'total_phi_common_assoc_pv': 0,
    'total_phi_track_pv_pass': 0,
    'total_phi_vertex_criteria_pass': 0,
}

kaon_source_counter = Counter()

for entry in range(tree.GetEntries()):
    snap = event_snapshot(tree, entry)
    gen_rows = snap['gen_rows']
    mu_rows = snap['mu_rows']
    phi_rows = snap['phi_rows']
    phi_candidates = sorted({row['phi_candidate'] for row in phi_rows})
    phi_by_candidate = {
        cand_idx: [row for row in phi_rows if row['phi_candidate'] == cand_idx]
        for cand_idx in phi_candidates
    }

    if gen_rows:
        summary['events_gen_nonempty'] += 1
    if mu_rows:
        summary['events_with_muons'] += 1
    if any(row['match_source'] == 'patRef' for row in mu_rows):
        summary['events_with_patref_match'] += 1
    if any(row['match_source'] == 'chi2Fallback' for row in mu_rows):
        summary['events_with_chi2_fallback'] += 1
    if phi_rows:
        summary['events_with_phi_candidates'] += 1
    if any(row['gen_match_idx'] >= 0 for row in phi_rows):
        summary['events_with_matched_phi_kaons'] += 1
    if any(rows[0]['phi_commonAssocPVPass'] == 1 for rows in phi_by_candidate.values()):
        summary['events_with_phi_common_assoc_pv'] += 1
    if any(rows[0]['phi_trackPVPass'] == 1 for rows in phi_by_candidate.values()):
        summary['events_with_phi_track_pv_pass'] += 1
    if any(rows[0]['phi_vertexCriteriaPass'] == 1 for rows in phi_by_candidate.values()):
        summary['events_with_phi_vertex_criteria_pass'] += 1

    summary['total_muons'] += len(mu_rows)
    summary['total_matched_muons'] += sum(row['match_idx'] >= 0 for row in mu_rows)
    summary['total_patref_matches'] += sum(row['match_source'] == 'patRef' for row in mu_rows)
    summary['total_chi2_fallback_matches'] += sum(row['match_source'] == 'chi2Fallback' for row in mu_rows)
    summary['total_phi_kaons'] += len(phi_rows)
    summary['total_matched_phi_kaons'] += sum(row['gen_match_idx'] >= 0 for row in phi_rows)
    summary['total_phi_candidates'] += len(phi_candidates)
    summary['total_phi_common_assoc_pv'] += sum(rows[0]['phi_commonAssocPVPass'] == 1 for rows in phi_by_candidate.values())
    summary['total_phi_track_pv_pass'] += sum(rows[0]['phi_trackPVPass'] == 1 for rows in phi_by_candidate.values())
    summary['total_phi_vertex_criteria_pass'] += sum(rows[0]['phi_vertexCriteriaPass'] == 1 for rows in phi_by_candidate.values())
    kaon_source_counter.update(row['gen_match_source'] for row in phi_rows)

for key, value in summary.items():
    print(f'{key}: {value}')
display(maybe_frame([
    {'kaon_match_source': key, 'count': value}
    for key, value in sorted(kaon_source_counter.items())
]))


entries: 945
events_gen_nonempty: 945
events_with_muons: 938
events_with_patref_match: 905
events_with_chi2_fallback: 33
events_with_phi_candidates: 5
events_with_matched_phi_kaons: 0
events_with_phi_common_assoc_pv: 0
events_with_phi_track_pv_pass: 0
events_with_phi_vertex_criteria_pass: 0
total_muons: 3857
total_matched_muons: 1948
total_patref_matches: 1914
total_chi2_fallback_matches: 34
total_phi_kaons: 12
total_matched_phi_kaons: 0
total_phi_candidates: 6
total_phi_common_assoc_pv: 0
total_phi_track_pv_pass: 0
total_phi_vertex_criteria_pass: 0


,kaon_match_source,count
0,unmatched,12


## Event Browser

Pick an event index and inspect the reco muons, fitted phi-daughter kaons, and the stored flat GEN collection for that event.

For kaons, `Phi_K_*_genMatchIdx` points into `MC_GenPart_*`, and `MC_GenPart_motherGenIdx` lets you walk from the matched kaon to its stored GEN phi when present.


In [11]:
ENTRY = first_entry_with_phi(tree)
snap = event_snapshot(tree, ENTRY)
print({key: snap[key] for key in ['entry', 'run', 'lumi', 'event', 'nMu']})
display(maybe_frame(snap['mu_rows']))
# display(maybe_frame(snap['phi_rows']))
# Full display of phi row. Transpose for better readability.
display(maybe_frame(snap['phi_rows']).T)
display(maybe_frame(snap['gen_rows']))


{'entry': 336, 'run': 1, 'lumi': 1, 'event': 331, 'nMu': 5}


,mu_idx,charge,pt,eta,phi,vertexId,match_idx,match_source,match_chi2,matched_pdg_id,matched_particle,matched_mother,matched_mother_idx,matched_gen_pt
0,0,-1,236.318457,-0.907819,0.575388,-1,-1,unmatched,-1.000000,NaN,None,None,NaN,NaN
1,1,-1,8.838431,1.266282,0.695050,2,10,patRef,1.064280,13.0,mu-,J/psi,6.0,8.724469
2,2,1,4.583950,-0.858517,-2.818235,2,13,patRef,0.388305,-13.0,mu+,J/psi,7.0,4.553099
3,3,-1,3.396563,-0.816002,-2.001315,2,12,patRef,0.381593,13.0,mu-,J/psi,7.0,3.380710
4,4,1,3.327698,0.702599,0.651823,2,11,patRef,1.379721,-13.0,mu+,J/psi,6.0,3.325838


,0,1
phi_candidate,0,0
kaon_role,K1,K2
track_idx,988,989
phi_fit_pass,-1,-1
phi_commonAssocPVPass,-1,-1
phi_commonAssocPVIdx,-1,-1
phi_trackPVPass,-1,-1
phi_vertexCriteriaPass,-1,-1
pt,2.056641,2.419922
eta,2.303171,2.244209


,gen_idx,handle_index,pdg_id,particle,status,mother_idx,mother_pdg_id,mother,pt,eta,phi,mass
0,0,4,443,J/psi,23,-1,21,21,7.810145,1.567656,0.593587,3.096900
1,1,5,443,J/psi,23,-1,21,21,7.810145,-0.919019,-2.548006,3.127869
2,2,10,443,J/psi,44,0,443,J/psi,7.802592,1.556925,0.577996,3.096900
3,3,11,443,J/psi,44,1,443,J/psi,8.007075,-0.904505,-2.385014,3.127869
4,4,12,443,J/psi,44,2,443,J/psi,7.819857,1.557796,0.573867,3.096900
5,5,13,443,J/psi,44,3,443,J/psi,7.896331,-0.919160,-2.337880,3.127869
6,6,14,443,J/psi,2,4,443,J/psi,12.048141,1.133795,0.683442,3.096900
7,7,15,443,J/psi,2,5,443,J/psi,7.296272,-0.899243,-2.473771,3.127869
8,8,22,333,phi,2,-1,2212,2212,4.899405,0.375109,3.137215,1.015448
9,9,23,333,phi,2,-1,2212,2212,1.646366,0.767992,2.607021,1.023406


In [ ]:
def candidate_muon_match_rows(tree, entry):
    snap = event_snapshot(tree, entry)
    tree.GetEntry(entry)
    candidate_specs = [
        ('Jpsi_1', getattr(tree, 'Jpsi_1_mu_1_Idx', []), getattr(tree, 'Jpsi_1_mu_2_Idx', [])),
        ('Jpsi_2', getattr(tree, 'Jpsi_2_mu_1_Idx', []), getattr(tree, 'Jpsi_2_mu_2_Idx', [])),
        ('Ups', getattr(tree, 'Ups_mu_1_Idx', []), getattr(tree, 'Ups_mu_2_Idx', [])),
    ]
    rows = []
    for label, idx1_vec, idx2_vec in candidate_specs:
        idx1_list = as_list(idx1_vec)
        idx2_list = as_list(idx2_vec)
        for cand_idx, (mu1_idx, mu2_idx) in enumerate(zip(idx1_list, idx2_list)):
            for role, mu_idx in (('daughter_1', int(mu1_idx)), ('daughter_2', int(mu2_idx))):
                if not (0 <= mu_idx < len(snap['mu_rows'])):
                    continue
                row = dict(snap['mu_rows'][mu_idx])
                row.update({
                    'candidate': label,
                    'candidate_index': cand_idx,
                    'role': role,
                })
                rows.append(row)
    return rows

def phi_candidate_kaon_rows(tree, entry):
    snap = event_snapshot(tree, entry)
    return snap['phi_rows']

display(maybe_frame(candidate_muon_match_rows(tree, ENTRY)))
display(maybe_frame(phi_candidate_kaon_rows(tree, ENTRY)))


,mu_idx,charge,pt,eta,phi,vertexId,match_idx,match_source,match_chi2,matched_pdg_id,matched_particle,matched_mother,matched_mother_idx,matched_gen_pt,candidate,candidate_index,role
0,1,-1,8.838431,1.266282,0.695050,2,10,patRef,1.064280,13,mu-,J/psi,6,8.724469,Jpsi_1,0,daughter_1
1,4,1,3.327698,0.702599,0.651823,2,11,patRef,1.379721,-13,mu+,J/psi,6,3.325838,Jpsi_1,0,daughter_2
2,2,1,4.583950,-0.858517,-2.818235,2,13,patRef,0.388305,-13,mu+,J/psi,7,4.553099,Jpsi_2,0,daughter_1
3,3,-1,3.396563,-0.816002,-2.001315,2,12,patRef,0.381593,13,mu-,J/psi,7,3.380710,Jpsi_2,0,daughter_2


,phi_candidate,kaon_role,track_idx,phi_fit_pass,phi_commonAssocPVPass,phi_commonAssocPVIdx,phi_trackPVPass,phi_vertexCriteriaPass,phi_maxAbsDzPV,phi_maxAbsDxyPV,...,dxyPV,dzAssocPV,dxyAssocPV,gen_match_idx,gen_match_source,gen_match_chi2,matched_particle,matched_phi_gen_idx,matched_phi_particle,gen_chain
0,0,K1,988,1,1,0,1,1,0.142656,0.018369,...,0.018369,-0.142656,0.018369,-1,unmatched,-1.0,None,-1,None,None
1,0,K2,989,1,1,0,1,1,0.142656,0.018369,...,0.002184,0.024082,0.002184,-1,unmatched,-1.0,None,-1,None,None


In [ ]:
def scan_decay_signatures(tree, max_events=None):
    pair_counter = Counter()
    event_counter = Counter()
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        pairs_in_event = set()
        for row in snap["gen_rows"]:
            key = (int(row["mother_pdg_id"]), int(row["pdg_id"]))
            pair_counter[key] += 1
            pairs_in_event.add(key)
        for key in pairs_in_event:
            event_counter[key] += 1

    rows = []
    for (mother, daughter), count in pair_counter.most_common():
        if abs(mother) not in {333, 443, 553} and abs(daughter) not in {13, 321, 333, 443, 553}:
            continue
        rows.append({
            "mother_pdg_id": mother,
            "mother": pdg_label(mother),
            "daughter_pdg_id": daughter,
            "daughter": pdg_label(daughter),
            "daughter_occurrences": count,
            "events_with_signature": event_counter[(mother, daughter)],
        })
    return rows

display(maybe_frame(scan_decay_signatures(tree, max_events=500)))


,mother_pdg_id,mother,daughter_pdg_id,daughter,daughter_occurrences,events_with_signature
0,443,J/psi,443,J/psi,5256,500
1,443,J/psi,13,mu-,1004,500
2,443,J/psi,-13,mu+,1004,500
3,21,21,443,J/psi,1000,500
4,333,phi,321,K+,689,500
5,333,phi,-321,K-,689,500
6,2212,2212,333,phi,669,486
7,421,421,-321,K-,24,24
8,-421,-421,321,K+,17,16
9,-431,-431,333,phi,7,7


In [ ]:
def match_source_breakdown(tree, max_events=None):
    source_counter = Counter()
    chi2_values = []
    phi_candidate_rows = []
    n_entries = tree.GetEntries() if max_events is None else min(tree.GetEntries(), max_events)

    for entry in range(n_entries):
        snap = event_snapshot(tree, entry)
        for row in snap["mu_rows"]:
            source_counter[row["match_source"]] += 1
            if row["match_source"] == "chi2Fallback" and row["match_chi2"] is not None:
                chi2_values.append(row["match_chi2"])

        seen_phi = set()
        for row in snap['phi_rows']:
            key = row['phi_candidate']
            if key in seen_phi:
                continue
            seen_phi.add(key)
            phi_candidate_rows.append({
                'entry': entry,
                'phi_candidate': key,
                'fit_pass': row['phi_fit_pass'],
                'commonAssocPVPass': row['phi_commonAssocPVPass'],
                'commonAssocPVIdx': row['phi_commonAssocPVIdx'],
                'trackPVPass': row['phi_trackPVPass'],
                'vertexCriteriaPass': row['phi_vertexCriteriaPass'],
                'maxAbsDzPV': row['phi_maxAbsDzPV'],
                'maxAbsDxyPV': row['phi_maxAbsDxyPV'],
            })

    rows = [{"source": key, "count": value} for key, value in sorted(source_counter.items())]
    display(maybe_frame(rows))
    if chi2_values:
        print({
            "fallback_matches": len(chi2_values),
            "chi2_min": min(chi2_values),
            "chi2_mean": sum(chi2_values) / len(chi2_values),
            "chi2_max": max(chi2_values),
        })
    display(maybe_frame(phi_candidate_rows))

match_source_breakdown(tree)


,source,count
0,chi2Fallback,34
1,patRef,1914
2,unmatched,1909


{'fallback_matches': 34, 'chi2_min': 0.11381018161773682, 'chi2_mean': 3.6815172977307262, 'chi2_max': 22.36978530883789}


,entry,phi_candidate,fit_pass,commonAssocPVPass,commonAssocPVIdx,trackPVPass,vertexCriteriaPass,maxAbsDzPV,maxAbsDxyPV
0,336,0,1,1,0,1,1,0.142656,0.018369
1,766,0,1,1,1,0,0,7.225317,0.470253
2,821,0,1,1,0,1,1,0.016885,0.000690
3,852,0,1,1,0,1,1,0.005781,0.002649
4,852,1,1,1,0,1,1,0.012129,0.004172
5,856,0,1,1,0,1,1,0.012979,0.003613


## MC Retention Switch Validation

This section compares the paired DPS validation ntuples produced with `ConfFile_cfg.py` on the same `10000`-event input slice:

- `RequireAcceptedCandidatesForMonteCarloTree=False`: keep every processed MC event and retain GEN info even when no stored candidate survives.
- `RequireAcceptedCandidatesForMonteCarloTree=True`: keep only events with at least one stored candidate, while preserving the same GEN and reco-matching content for those kept events.


In [ ]:
def open_tree_for_path(path):
    if not path.exists():
        return None, None
    root_handle = ROOT.TFile.Open(str(path), 'READ')
    if not root_handle or root_handle.IsZombie():
        return None, None
    tree_handle = root_handle.Get(TREE_PATH)
    if not tree_handle:
        root_handle.Close()
        return None, None
    return root_handle, tree_handle

retention_handles = {}
retention_trees = {}
retention_rows = []
for label, path in RETENTION_VALIDATION_NTUPLES.items():
    handle, tree_handle = open_tree_for_path(path)
    if tree_handle is None:
        retention_rows.append({
            'sample': label,
            'path': str(path),
            'available': False,
        })
        continue
    retention_handles[label] = handle
    retention_trees[label] = tree_handle
    retention_rows.append({
        'sample': label,
        'path': str(path),
        'available': True,
        'entries': int(tree_handle.GetEntries()),
        'events_gen_nonempty': int(tree_handle.GetEntries('Length$(MC_GenPart_pdgId)>0')),
        'events_with_candidates': int(tree_handle.GetEntries('Length$(Pri_VtxProb)>0')),
        'events_without_candidates': int(tree_handle.GetEntries('Length$(Pri_VtxProb)==0')),
    })

display(maybe_frame(retention_rows))


In [ ]:
def event_key_counter(tree, require_candidates=None):
    counter = Counter()
    for entry in range(tree.GetEntries()):
        snap = event_snapshot(tree, entry)
        has_candidate = bool(snap['phi_rows']) or len(as_list(getattr(tree, 'Pri_VtxProb', []))) > 0
        if require_candidates is True and not has_candidate:
            continue
        if require_candidates is False and has_candidate:
            continue
        counter[(snap["run"], snap["lumi"], snap["event"])] += 1
    return counter

def muon_match_counts(tree, require_candidates=None):
    counter = Counter()
    for entry in range(tree.GetEntries()):
        snap = event_snapshot(tree, entry)
        has_candidate = bool(snap['phi_rows']) or len(as_list(getattr(tree, 'Pri_VtxProb', []))) > 0
        if require_candidates is True and not has_candidate:
            continue
        if require_candidates is False and has_candidate:
            continue
        counter.update(row["match_source"] for row in snap["mu_rows"])
    return counter

false_tree = retention_trees.get('keep_all_mc_events')
true_tree = retention_trees.get('require_candidates')
if false_tree and true_tree:
    false_candidate_counter = event_key_counter(false_tree, require_candidates=True)
    true_counter = event_key_counter(true_tree)
    missing_from_true = false_candidate_counter - true_counter
    extra_in_true = true_counter - false_candidate_counter
    print({
        'false_candidate_entries': int(sum(false_candidate_counter.values())),
        'true_entries': int(sum(true_counter.values())),
        'distinct_false_candidate_keys': int(len(false_candidate_counter)),
        'distinct_true_keys': int(len(true_counter)),
        'missing_entry_multiplicity': int(sum(missing_from_true.values())),
        'extra_entry_multiplicity': int(sum(extra_in_true.values())),
    })

    false_mu_counter = muon_match_counts(false_tree, require_candidates=True)
    true_mu_counter = muon_match_counts(true_tree)
    display(maybe_frame([
        {
            'source': source,
            'false_candidate_subset': int(false_mu_counter.get(source, 0)),
            'true_tree': int(true_mu_counter.get(source, 0)),
        }
        for source in sorted(set(false_mu_counter) | set(true_mu_counter))
    ]))
else:
    print('Retention validation ntuples are not both available; skip this section or update RETENTION_VALIDATION_NTUPLES.')


## Suggested Follow-ups

- Change `ENTRY` only if you want to inspect a different signal-like event; by default the notebook now picks the first entry with a stored `phi` candidate.
- Point `RETENTION_VALIDATION_NTUPLES` at a different false/true ntuple pair if you want to repeat the MC-retention validation on another sample or cut set.
- Filter `phi_rows` by `phi_vertexCriteriaPass == 1` or `passTrackPV == 0` to study which kaon-side PV requirement is failing.
- Compare `phi_commonAssocPVIdx`, `vertexId`, `dzPV`, and `dxyPV` against `Pri_*` acceptance diagnostics when studying why combined candidates fail or survive.
- Filter the GEN table by `matched_phi_gen_idx` to inspect only kaons that trace back to stored GEN `phi` ancestors.
